In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [4]:
# images => scale(0,1) => normalize(-1,+1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset = CIFAR10(root = "CNN_minor_project/data",train= True,download = True,transform = transform)
testset = CIFAR10(root = "CNN_minor_project/data",train = False,download =True,transform =  transform)


C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
train_loader = DataLoader(trainset,batch_size = 64,shuffle = True)
test_loader = DataLoader(testset,batch_size = 64)


In [6]:
# build our CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers =  nn.Sequential(
            nn.Conv2d(3,32,kernel_size = 3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),# kernal_size and  stride
            nn.Conv2d(32,64,kernel_size = 3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,kernel_size = 3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layer(x)
        return x
        

In [7]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
### training CNN
training_loss = []
validation_loss = []
epochs =  10
for epoch in range(epochs):
    model.train()
    epochs_loss = 0.0
    for images,labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()
        epochs_loss += loss.item()
    loss_total = epochs_loss/len(train_loader)
    training_loss.append(loss_total)
    print(f"{epoch+1}/{epochs} loss is {loss_total}\n")

    model.eval()
    with  torch.no_grad():
            running_val_loss = 0.0
            for images,labels in test_loader:
                output = model(images)
                loss = criterion(output,labels)
                running_val_loss += loss.item()
    loss_val_total = running_val_loss / len(test_loader)
    validation_loss.append(loss_val_total)
    print(f"{epoch+1}/{epochs} loss is {loss_total} and validation loss is {loss_val_total}")

1/10 loss is 1.3626918962696934

1/10 loss is 1.3626918962696934 and validation loss is 1.0518245199683365
2/10 loss is 0.9194804892668029

2/10 loss is 0.9194804892668029 and validation loss is 0.8437286972240278
3/10 loss is 0.7351770424629416

3/10 loss is 0.7351770424629416 and validation loss is 0.7805733815499931
4/10 loss is 0.608300891831098

4/10 loss is 0.608300891831098 and validation loss is 0.7402770253503399
5/10 loss is 0.5046786535960024

5/10 loss is 0.5046786535960024 and validation loss is 0.7993516100060408
6/10 loss is 0.4149498272772945

6/10 loss is 0.4149498272772945 and validation loss is 0.7419741721289932
7/10 loss is 0.32783225107261593

7/10 loss is 0.32783225107261593 and validation loss is 0.8333501403878449
8/10 loss is 0.2557677423862545

8/10 loss is 0.2557677423862545 and validation loss is 0.8552340313697316
9/10 loss is 0.197157556043409

9/10 loss is 0.197157556043409 and validation loss is 0.9764435167904872
10/10 loss is 0.15857680342005342

10/1

In [10]:
# evaluate
correct_labels = 0.0
total_labels = 0.0
model.eval()
with torch.no_grad():
    for images,labels in test_loader:
        output = model.forward(images)
        _,predicted = torch.max(output,1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print("total is : ",total_labels)
print("correct is  : ",correct_labels)
print("accuracy: ", correct_labels/total_labels * 100)       

total is :  10000.0
correct is  :  7616.0
accuracy:  76.16000000000001
